# 🔻 Churn Prediction — Random Forest Classifier
> **Caso de negocio:** Financiera Efectiva · Retención de Cartera
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Financiera Efectiva gestiona una cartera de ~50,000 clientes de crédito de consumo. El equipo de retención solo puede contactar al **15% de la cartera por mes**, pero hoy actúa de forma reactiva: recién se entera cuando el cliente ya dejó de operar o cerró sus productos.

**Objetivo:** Predecir qué clientes tienen mayor probabilidad de abandonar (dejar de operar) sus productos en los próximos 30 días, para activar retención proactiva.

**KPIs de éxito:**
- Reducir la tasa de churn mensual de 8% a 5%
- El top 20% de score de riesgo debe capturar 60%+ de los churners reales
- Costo de retención por cliente < costo de adquisición de un cliente nuevo

**Algoritmo:** Random Forest Classifier (ensemble de árboles con bagging)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q shap plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
# Definición formal del problema
PROBLEMA = {
    'tipo': 'Clasificación binaria supervisada',
    'target': 'churn (0=cliente activo, 1=cliente en fuga)',
    'ventana_prediccion': '30 días',
    'features': [
        'dias_inactivo  → días desde la última transacción',
        'quejas         → reclamos registrados en los últimos 6 meses',
        'logins_30d     → ingresos a la app/banca digital en 30 días',
        'saldo_prom_k   → saldo promedio en cuentas (S/. miles)',
        'productos      → número de productos activos (créditos, cuentas, seguros)',
    ],
    'criterios_aceptacion': {
        'AUC-ROC': '>= 0.75',
        'Recall':  '>= 0.65  (minimizar clientes que se fugan sin que actuemos)',
        'Precision': '>= 0.55',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con query a BigQuery (eventos de banca digital + core bancario)
N = 1200

dias_inactivo = np.random.randint(0, 91, N)
quejas        = np.where(np.random.random(N) < 0.25,
                          np.random.randint(1, 6, N), 0)
logins_30d    = np.random.randint(0, 31, N)
saldo_prom_k  = np.round(np.random.uniform(1, 50, N), 1)
productos     = np.random.randint(1, 6, N)
antiguedad_m  = np.random.randint(1, 73, N)
edad          = np.random.randint(22, 67, N)

# DGP: proceso generador de datos con relaciones realistas
z = (-1.8
     + 0.05  * dias_inactivo   # más días sin operar → más riesgo
     + 0.50  * quejas          # cada reclamo eleva el riesgo de fuga
     - 0.10  * logins_30d      # más uso de la app → más vinculación
     - 0.03  * saldo_prom_k    # más saldo → más costoso migrar de banco
     - 0.25  * productos       # más productos → más vinculación
     + np.random.normal(0, 0.8, N))

prob  = 1 / (1 + np.exp(-z))
churn = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'cliente_id':     range(1, N+1),
    'dias_inactivo':  dias_inactivo,
    'quejas':         quejas,
    'logins_30d':     logins_30d,
    'saldo_prom_k':   saldo_prom_k,
    'productos':      productos,
    'antiguedad_m':   antiguedad_m,
    'edad':           edad,
    'churn':          churn
})

print(f'Dataset: {df.shape}')
print(f'\nDistribución del target:')
print(df['churn'].value_counts(normalize=True).map('{:.1%}'.format))
df.head()

In [ ]:
# Estadísticas descriptivas por clase
print('=== FUGADOS (churn=1) ===')
display(df[df['churn']==1].describe().round(2))
print('\n=== RETENIDOS (churn=0) ===')
display(df[df['churn']==0].describe().round(2))

In [ ]:
# Visualización: distribuciones por clase
features = ['dias_inactivo', 'quejas', 'logins_30d', 'saldo_prom_k', 'productos']
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f for f in features])

colors = {0: '#0d9488', 1: '#c0392b'}
for idx, feat in enumerate(features):
    r, c = idx // 3 + 1, idx % 3 + 1
    for clase, label in [(0, 'Retenido'), (1, 'Fugado')]:
        data = df[df['churn'] == clase][feat]
        fig.add_trace(
            go.Histogram(x=data, name=label, opacity=0.65,
                         marker_color=colors[clase],
                         showlegend=(idx == 0)),
            row=r, col=c
        )

fig.update_layout(barmode='overlay', height=500,
                  title='Distribución de features por clase del target',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de correlaciones
num_cols = ['dias_inactivo', 'quejas', 'logins_30d', 'saldo_prom_k',
            'productos', 'antiguedad_m', 'edad', 'churn']
corr = df[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Correlación entre variables')
fig.update_layout(height=450, paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['dias_inactivo', 'quejas', 'logins_30d', 'saldo_prom_k', 'productos']
TARGET   = 'churn'

X = df[FEATURES].copy()
y = df[TARGET].astype(int)

# Feature engineering
X['actividad_score'] = X['logins_30d'] / (X['dias_inactivo'] + 1)   # uso digital vs inactividad
X['vinculacion']     = X['productos'] * X['saldo_prom_k'] / 10      # proxy de relación con el banco

print('Features finales:', X.columns.tolist())
print('Nulos:', X.isnull().sum().sum())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')
print(f'Tasa de churn train: {y_train.mean():.1%}')
print(f'Tasa de churn test:  {y_test.mean():.1%}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Random Forest: ensemble de árboles independientes con votación mayoritaria
# class_weight='balanced': compensa el desbalance (menos clientes fugados que activos)
rf = RandomForestClassifier(
    n_estimators=200,        # número de árboles
    max_depth=6,             # profundidad — evita overfitting
    class_weight='balanced', # penaliza más los errores en la clase minoritaria
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Validación cruzada — estimación honesta del performance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
cv_rec = cross_val_score(rf, X_train, y_train, cv=cv, scoring='recall')

print(f'CV AUC-ROC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'CV Recall:  {cv_rec.mean():.3f} ± {cv_rec.std():.3f}')

In [ ]:
# Predicciones en test set
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

# Importancia de variables
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(fi, x='importance', y='feature', orientation='h',
             title='Importancia de variables (Random Forest)',
             color='importance', color_continuous_scale='reds',
             text=fi['importance'].map('{:.3f}'.format))
fig.update_traces(textposition='outside')
fig.update_layout(height=380, yaxis={'categoryorder': 'total ascending'},
                  plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False)
fig.show()

## 5️⃣ Fase 5 — Evaluation

In [ ]:
# Métricas en test set
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall':    recall_score(y_test, y_pred, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred, zero_division=0),
    'AUC-ROC':   roc_auc_score(y_test, y_prob),
}

criterios = {'AUC-ROC': 0.75, 'Recall': 0.65, 'Precision': 0.55}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral {umbral:.2f})'
    print(f'  {k:12s}: {v:.4f}  {estado}')

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                          name=f'RF (AUC={auc:.3f})',
                          line=dict(color='#1a4c8c', width=2.5)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Random', line=dict(color='#ccc', dash='dash')))
fig.update_layout(title='Curva ROC — Churn Prediction',
                  xaxis_title='Tasa Falsos Positivos', yaxis_title='Tasa Verdaderos Positivos',
                  height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['Retenido', 'Fugado'])
disp.plot(ax=ax, colorbar=False, cmap='Reds')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

print('\nInterpretación:')
tn, fp, fn, tp = cm.ravel()
print(f'  Verdaderos Positivos (VP): {tp} — fugas detectadas a tiempo')
print(f'  Falsos Negativos    (FN): {fn} — clientes que se fugan sin alerta (costo alto)')
print(f'  Falsos Positivos    (FP): {fp} — contactamos a clientes que igual se quedaban')
print(f'  Verdaderos Negativos(VN): {tn} — correctamente identificados como activos')

In [ ]:
# Análisis de lift: cuánto mejor es el modelo vs selección aleatoria
df_test = pd.DataFrame({'prob': y_prob, 'real': y_test.values})
df_test = df_test.sort_values('prob', ascending=False).reset_index(drop=True)

deciles = []
n = len(df_test)
for d in range(10):
    subset = df_test.iloc[d*n//10:(d+1)*n//10]
    churn_rate = subset['real'].mean()
    deciles.append({'decil': f'D{d+1}', 'tasa_churn': churn_rate})

df_lift = pd.DataFrame(deciles)
baseline = df_test['real'].mean()
df_lift['lift'] = df_lift['tasa_churn'] / baseline

fig = px.bar(df_lift, x='decil', y='lift',
             title=f'Lift por decil (baseline = {baseline:.1%})',
             text=df_lift['lift'].map('{:.2f}x'.format),
             color='lift', color_continuous_scale='reds')
fig.add_hline(y=1, line_dash='dash', line_color='gray',
              annotation_text='Baseline (sin modelo)')
fig.update_traces(textposition='outside')
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white',
                  showlegend=False)
fig.show()

top2_capture = df_lift['tasa_churn'][:2].sum() * (n // 10) / df_test['real'].sum()
print(f'\nEl top 20% de clientes por score concentra {top2_capture:.0%} de los churners reales')

In [ ]:
# SHAP: por qué el modelo predice fuga para un cliente específico
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Waterfall del cliente con mayor riesgo de fuga
cliente_riesgo_idx = int(y_prob.argmax())
print(f'Cliente con mayor riesgo de fuga: {y_prob[cliente_riesgo_idx]:.1%}')
print(df.iloc[X_test.index[cliente_riesgo_idx]][FEATURES + ['churn']])

# shap >= 0.45 devuelve un array (n_samples, n_features, n_clases);
# versiones previas devuelven una lista de arrays por clase
if isinstance(shap_values, list):
    sv = shap_values[1][cliente_riesgo_idx]
    base_value = explainer.expected_value[1]
else:
    sv = shap_values[cliente_riesgo_idx, :, 1]
    base_value = explainer.expected_value[1]

shap.waterfall_plot(
    shap.Explanation(
        values=sv,
        base_values=base_value,
        data=X_test.iloc[cliente_riesgo_idx],
        feature_names=X_test.columns.tolist()
    )
)

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Exportar score de riesgo sobre toda la cartera
X_all = df[FEATURES].copy()
X_all['actividad_score'] = X_all['logins_30d'] / (X_all['dias_inactivo'] + 1)
X_all['vinculacion']     = X_all['productos'] * X_all['saldo_prom_k'] / 10

df['score_churn'] = (rf.predict_proba(X_all)[:, 1] * 100).round(1)
df['riesgo'] = pd.cut(
    df['score_churn'],
    bins=[0, 35, 65, 100],
    labels=['Riesgo bajo', 'Riesgo medio', 'Riesgo alto']
)

print('Distribución de riesgo:')
print(df['riesgo'].value_counts())
print(f'\nClientes a contactar (Riesgo alto): {(df["riesgo"]=="Riesgo alto").sum()}')
df[['cliente_id','score_churn','riesgo']].sort_values('score_churn', ascending=False).head(10)

In [ ]:
# Guardar CSV para integración con CRM de retención
df[['cliente_id','score_churn','riesgo']].to_csv('scores_churn_financiera.csv', index=False)
print('Archivo scores_churn_financiera.csv guardado ✓')

# Guardar modelo
import joblib
joblib.dump(rf, 'modelo_churn_financiera.pkl')
print('Modelo modelo_churn_financiera.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo de ROI
n_clientes      = len(df)
n_riesgo_alto   = (df['riesgo'] == 'Riesgo alto').sum()
churn_rate_alto = df[df['riesgo']=='Riesgo alto']['churn'].mean()
churn_rate_base = df['churn'].mean()
costo_retencion = 25     # S/. por gestión de retención (llamada + oferta)
valor_cliente   = 1200   # S/. valor anual promedio de un cliente retenido

churners_capturados = int(n_riesgo_alto * churn_rate_alto)
ahorro_estimado     = churners_capturados * valor_cliente - n_riesgo_alto * costo_retencion
lift_riesgo         = churn_rate_alto / churn_rate_base

print('=== RESUMEN EJECUTIVO ===')
print(f'Clientes en cartera:                   {n_clientes:,}')
print(f'Clientes en riesgo alto (a contactar): {n_riesgo_alto:,} ({n_riesgo_alto/n_clientes:.0%})')
print(f'Tasa de churn en riesgo alto:           {churn_rate_alto:.1%}')
print(f'Tasa de churn cartera total:            {churn_rate_base:.1%}')
print(f'Lift de concentración de riesgo:        {lift_riesgo:.1f}x')
print(f'Churners potencialmente retenidos:      ~{churners_capturados}')
print(f'Valor estimado retenido (S/.):          S/.{ahorro_estimado:,.0f}')
print(f'\nArquitectura de producción:')
print('  Core bancario + app → BigQuery → score diario → CRM retención → gestor de cuenta')